In [1]:
import polars as pl
import json
import os
from __future__ import annotations
import json
from typing import Any, Dict, List

In [2]:
## función para leer json en local

def leer_json(path:str) -> Dict[str, Any]:

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
    
    

In [3]:
json = leer_json("/Users/danielsanchezcuenca/Desktop/GCP/redelectrica-proyecto/raw/ire.json")

In [4]:
print(json)

{'requests': {'start_date': '2018-01-01T00:00', 'end_date': '2018-12-31T23:59', 'time_trunc': 'month', 'geo_trunc': 'electric_system', 'geo_limit': 'peninsular', 'geo_ids': '8741'}, 'data': {'data': {'type': 'Índice de Red Eléctrica General (IRE)', 'id': 'dem4', 'attributes': {'title': 'Índice de Red Eléctrica General (IRE)', 'last-update': '2022-03-23T10:46:36.000+01:00', 'description': 'Información adelantada de la evolución del consumo eléctrico del conjunto de empresas que tienen una potencia contratada superior a 450 kW. Índice 2010=100 y % de crecimiento sobre el mismo periodo del año anterior'}, 'meta': {'cache-control': {'cache': 'HIT', 'expireAt': '2025-09-28T11:50:42'}}}, 'included': [{'type': 'Índice general', 'id': '1596', 'groupId': None, 'attributes': {'title': 'Índice general', 'description': None, 'color': '#cdcdcd', 'icon': None, 'type': 'invertedAxis', 'magnitude': None, 'composite': False, 'last-update': '2022-03-23T10:46:36.000+01:00', 'values': [{'value': 124.345, 

In [7]:
import polars as pl
from typing import Any, Dict, List

def aplanar_ire(raw: Dict[str, Any]) -> pl.DataFrame:
    rows: List[Dict[str, Any]] = []

    # --- Categoría padre (p. ej. "Generación por tecnología") ---
    parent = raw.get("data", {}).get("data", {})  # en tu JSON es un dict
    parent_attrs = parent.get("attributes", {}) if isinstance(parent, dict) else {}
    parent_category = parent_attrs.get("title") or parent.get("type") or "desconocido"
    parent_id = parent.get("id")

    # --- Función recursiva por si existen hijos en 'attributes.content' ---
    def visita_nodo(nodo: Dict[str, Any]):
        attrs = nodo.get("attributes", {})
        sub_type = attrs.get("title") or nodo.get("type")        # p.ej. "Hidráulica"
        energy_type = attrs.get("type")                           # p.ej. "Renovable"/"No-Renovable"

        # Caso hoja: values
        for v in attrs.get("values", []):
            rows.append({
                "dt": v.get("datetime"),
                "value": v.get("value"),
                "percentage": v.get("percentage"),
                "energy_type": energy_type,
                "sub_type": sub_type,
                "parent_category": parent_category,  # << padre
                "parent_id": parent_id,
                "source": "ree/balance",
            })

        # Caso intermedio: hijos en content
        for hijo in attrs.get("content", []):
            visita_nodo(hijo)

    # --- Recorremos los nodos principales ---
    for nodo in raw.get("data", {}).get("included", []):
        visita_nodo(nodo)

    # --- DataFrame tipado ---
    df = pl.DataFrame(rows).with_columns(
        pl.col("dt").str.strptime(pl.Datetime, strict=False),
        pl.col("value").cast(pl.Float64, strict=False),
        pl.col("percentage").cast(pl.Float64, strict=False),
        pl.col("energy_type").cast(pl.String),
        pl.col("sub_type").cast(pl.String),
        pl.col("parent_category").cast(pl.String),
        pl.col("parent_id").cast(pl.String, strict=False),
        pl.col("source").cast(pl.String),
    )
    return df


In [8]:
df = aplanar_ire(json)
df.head()

dt,value,percentage,energy_type,sub_type,parent_category,parent_id,source
"datetime[μs, UTC]",f64,f64,str,str,str,str,str
2017-12-31 23:00:00 UTC,124.345,0.541591,"""invertedAxis""","""Índice general""","""Índice de Red Eléctrica Genera…","""dem4""","""ree/balance"""
2018-01-31 23:00:00 UTC,117.039,0.516996,"""invertedAxis""","""Índice general""","""Índice de Red Eléctrica Genera…","""dem4""","""ree/balance"""
2018-02-28 23:00:00 UTC,126.897,0.539574,"""invertedAxis""","""Índice general""","""Índice de Red Eléctrica Genera…","""dem4""","""ree/balance"""
2018-03-31 22:00:00 UTC,124.703,0.534465,"""invertedAxis""","""Índice general""","""Índice de Red Eléctrica Genera…","""dem4""","""ree/balance"""
2018-04-30 22:00:00 UTC,132.531,0.547588,"""invertedAxis""","""Índice general""","""Índice de Red Eléctrica Genera…","""dem4""","""ree/balance"""


In [9]:
df = df.unique("sub_type")
print(df)

shape: (4, 8)
┌────────────┬──────────┬────────────┬────────────┬────────────┬───────────┬───────────┬───────────┐
│ dt         ┆ value    ┆ percentage ┆ energy_typ ┆ sub_type   ┆ parent_ca ┆ parent_id ┆ source    │
│ ---        ┆ ---      ┆ ---        ┆ e          ┆ ---        ┆ tegory    ┆ ---       ┆ ---       │
│ datetime[μ ┆ f64      ┆ f64        ┆ ---        ┆ str        ┆ ---       ┆ str       ┆ str       │
│ s, UTC]    ┆          ┆            ┆ str        ┆            ┆ str       ┆           ┆           │
╞════════════╪══════════╪════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╡
│ 2017-12-31 ┆ 105.247  ┆ 0.452882   ┆ invertedAx ┆ Índice     ┆ Índice de ┆ dem4      ┆ ree/balan │
│ 23:00:00   ┆          ┆            ┆ is         ┆ general    ┆ Red       ┆           ┆ ce        │
│ UTC        ┆          ┆            ┆            ┆ corregido  ┆ Eléctrica ┆           ┆           │
│            ┆          ┆            ┆            ┆            ┆ Genera…   ┆ 

In [10]:
df_unico = df.unique(subset=["dt","energy_type","sub_type","parent_category"])

In [11]:
df_clean = df_unico.filter(
    ~ pl.all_horizontal(pl.all().is_null())
)

In [12]:
def normalize_nulls(df: pl.DataFrame) -> pl.DataFrame:

    out = df

    for col,dtype in df.schema.items():
        if dtype in(pl.Float64, pl.Float32):
            out = out.with_columns(pl.col(col).fill_nan(None))
        if dtype in (pl.String,pl.Utf8):
            out = out.with_columns(pl.when(pl.col(col).str.strip_chars()=="")
                                   .then(None)
                                   .otherwise(pl.col(col))
                                   .alias(col)
                                   )
    return out

In [13]:
df_clean = normalize_nulls(df)